# System Prompts (Gemini Counterpart)

In Google Gemini, system prompts are passed via `config={"system_instruction": ...}`.

### Case Study: Math Tutor Specialist
- **Goal**: Guide a student to solve math equations without directly giving away the answer.
- **Responses should**:
  - Initially only give hints
  - Patiently walk the student through step-by-step
  - Show similar problem examples
- **Responses should NOT**:
  - Immediately give a direct answer
  - Tell the student to use a calculator

In [ ]:
# Install dependencies
%pip install google-genai python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Initialize Gemini Client
import os
import logging
from google import genai

# Suppress AFC informational notices
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
model = "gemini-3.5-flash-lite"

## 1. Initial Test (Without System Prompt)
Without a system prompt, Gemini immediately solves the equation completely for the user.

In [ ]:
# Helper functions (basic version)
def add_user_message(messages, text):
    user_message = {"role": "user", "parts": [{"text": text}]}
    messages.append(user_message)

def add_model_message(messages, text):
    model_message = {"role": "model", "parts": [{"text": text}]}
    messages.append(model_message)

def chat(messages):
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config={"max_output_tokens": 1000},
    )
    return response.text

In [ ]:
messages = []
add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages)
print(answer.strip())

## 2. Refactoring `chat()` for Dynamic System Prompts in Gemini

In Gemini, system instructions are placed inside the `config` object under `system_instruction`.

In [ ]:
def chat(messages, system=None):
    config = {
        "max_output_tokens": 1000
    }

    if system:
        config["system_instruction"] = system

    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=config,
    )
    return response.text

## 3. Testing with the Math Tutor System Prompt
Now Gemini adopts the patient math tutor persona, asking guiding questions instead of calculating the answer directly!

In [ ]:
messages = []

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system=system)
print(answer.strip())